# Qwen3-Reranker-0.6B — DIMER E2E reranker fine-tuning tutorial: intent shortlists on Banking77 (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/qwen3-reranker-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/qwen3-reranker-pipeline/blob/main/tutorials/qwen3_reranker_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-Qwen%2FQwen3--Reranker--0.6B-ffcc4d?style=flat)](https://huggingface.co/Qwen/Qwen3-Reranker-0.6B) [![Upstream](https://img.shields.io/badge/Upstream-QwenLM%2FQwen3--Embedding-181717?style=flat&logo=github&logoColor=white)](https://github.com/QwenLM/Qwen3-Embedding) [![arXiv](https://img.shields.io/badge/arXiv-2506.05176-b31b1b.svg)](https://arxiv.org/abs/2506.05176)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** pointwise query-document relevance reranking and bounded listwise fine-tuning of the last decoder layers on query / positive / negatives records, measured by held-out ranking recall@k and MRR, using the pinned Qwen3-Reranker-0.6B weights

**This notebook is standalone.** It carries the repository's package (3 modules under `src/qwen3_reranker_pipeline/`, at revision `a2066cac44b0`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the Hugging Face Hub at the immutable revision `e61197ed45024b0ed8a2d74b80b4d909f1255473` (~1207 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned Qwen3-Reranker-0.6B snapshot (safetensors, 1.19 GB), fetches the two digest-pinned Banking77 CSV files from the project repository (1.1 MB, no credential), pairs every customer message with its intent phrase and a seeded shortlist of five other intent phrases and draws 231 / 77 / 154 training, validation and test records balanced over the 77 intents from the release's own partition, reranks one test shortlist through the inference contract with an input manifest and a rejection probe, scores the frozen reranker on the test shortlists by recall@1 / recall@3 / recall@5 and MRR beside the random floor and a lexical (token-overlap) baseline, runs a bounded listwise fine-tuning of the last two decoder layers with validation-MRR epoch selection, scores the held-out split again, reranks the same shortlists with the adapted model, exports the adapter as safetensors with a manifest, and reloads that artifact into a fresh pipeline to verify parity. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5). On CPU the whole path takes about fifteen minutes of model time after the downloads — a cross-encoder scores every query–candidate pair with a full forward pass, so evaluation dominates; a CUDA runtime is used automatically when present (bfloat16 there, float32 on CPU).

**Bring Your Own Data:** After the tutorial workflow completes, set `USE_BYOD = True` in Section 4 and re-run from that cell to supply your own records as a CSV (columns `id`, `query`, `positive`, `negatives` with the negatives separated by ` | `), a JSON array or a JSONL file of `{{id, query, positive, negatives}}` records; set `INSTRUCTION` to a one-line description of your relevance judgement. They pass through the same validation, seeded query-disjoint split, floor and baseline, frozen scoring, fine-tuning, held-out evaluation, reranking, artifact export and reload-parity cells as the Banking77 sample. The expected schema and the ceilings are stated in the Prerequisites and in Section 4, and uploaded files stay inside this runtime. BYOD is optional and never part of the default path.

At inference each `(query, document)` pair is wrapped in the fixed upstream system/user/assistant prompt together with an instruction, one forward pass of the causal language model reads the last-position logits of the `yes` and `no` tokens, and a two-way softmax turns them into a **relevance score** in [0, 1]: the `yes` share. **The score is not a calibrated probability**, no threshold is shipped, and the `ranking` the pipeline returns is an ordering of the supplied pairs, not an acceptance decision — the caller owns any cut-off. What the upstream checkpoint supplies is the model, tokenizer and prompt convention; what the carried pipeline module adds is manifest verification, input validation and ceilings, prompt assembly, the two-logit read-out, a fixed output contract and the `validate_inputs` and `evaluation_report` stage helpers. Free-text generation is deliberately not reachable through this package.

What this notebook adds to inference is **adaptation measured by ranking**. The dataset is real: Banking77 (Casanueva et al., 2020; CC BY 4.0) ships 13,083 customer-support messages labelled with 77 fine-grained banking intents as two digest-pinned CSV files fetched from the project repository at a pinned commit. Every intent name becomes a short **document** (`card_arrival` → `card arrival`), every message a **query** whose positive is its intent phrase, and every query carries a seeded **shortlist** of five other intent phrases — the three with the highest token overlap with the message plus two random ones — so reranking the six-entry shortlist is intent detection over hard candidates, a task the reranker was never tuned for. The carried `metrics.py` orders each shortlist by the relevance score and reads the rank of the positive: **recall@1**, **recall@3**, **recall@5** and **MRR**, ties counted against the positive; a **random floor** (1 / 6 recall@1) and a **lexical baseline** (Jaccard token overlap between message and phrase — the very signal the hard negatives were chosen by) frame the frozen number. The fine-tuning question is whether a bounded listwise adaptation of the last decoder layers on 231 shortlists raises ranking on messages the model has not seen. Nothing here is a quality claim about your reranking task: it is one seeded split of one corpus.

**Learning objectives:** install the pinned runtime; read what the carried pipeline, metrics and dataset modules guarantee; stage and digest-verify the immutable upstream snapshot; fetch a digest-pinned real corpus and validate and split it without leakage; rerank a shortlist through the public API with the instruction contract and read the score and ranking contract correctly; read recall@k and MRR beside a random floor and a lexical baseline and understand what they do and do not measure; run a bounded listwise fine-tuning with explicit hyperparameters and validation-based epoch selection; evaluate on an independent test split; compare rankings before and after; and export a safetensors adapter that reloads against the pinned base with verified parity.

**This notebook does not demonstrate:** embedding or first-stage retrieval (see the sibling Qwen3 embedding pipeline), text generation or chat, calibrated probabilities or a relevance threshold, graded (non-binary) relevance, hard-negative mining beyond the seeded shortlists of the dataset contract, full-model or embedding-table training, and any claim that a Banking77 intent shortlist stands in for your reranking task. The repository exposes none of these.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU and uses CUDA automatically when available. **Precision differs by device:** the pipeline runs float32 on CPU and bfloat16 on CUDA, so scores and the recorded metrics can differ between the two. CPU is adequate but not fast for a cross-encoder: the build record measured about 5 s to load and digest-verify the 1.19 GB snapshot, about 132 s to score the 154 test shortlists (924 pairs) and about 170 s per training epoch over 231 shortlists plus a validation pass per epoch. The pinned `torch==2.14.0` install and the 1.19 GB checkpoint are the large downloads of the run.
- **Knowledge:** basic Python; what a cross-encoder does and why it scores one pair per forward pass; what recall@k and mean reciprocal rank measure over a candidate list; what a listwise softmax loss does.
- **Data contract:** records are `{{id, query, positive, negatives}}` — a query of 1..100,000 characters, a positive document of 1..1,000 characters and 1..15 distinct negative documents (the pair prompt is truncated longest-first to 8,192 tokens at inference and to 192 tokens **during training only**), ids matching `[A-Za-z0-9_.:-]{{1,64}}` and unique; a dataset needs 8..20,000 records; queries are de-duplicated case-insensitively before splitting so the same message never sits in two splits. BYOD accepts CSV (negatives separated by ` | `), JSON or JSONL in that shape.
- **Validation is structural, not semantic:** nothing checks that a positive is relevant to its query, that the negatives are not, or that the instruction describes the judgement — a mislabelled record set is fine-tuned on without complaint.
- **Privacy:** Do not upload confidential or restricted data to a hosted runtime unless you are authorized to process it there — an internal query log with its relevance labels is exactly that. The default path uploads nothing.
- **External access (data):** besides the Hub, the default path fetches two pinned objects (`train.csv` 839,073 bytes, `test.csv` 239,961 bytes; SHA-256 `b06e26ac…` / `d12d6e3b…`) from `raw.githubusercontent.com` at the pinned `PolyAI-LDN/task-specific-datasets` commit over HTTPS, each refused on any mismatch before it is read; Banking77 is CC BY 4.0 (Casanueva et al., 2020).
- **External access:** the Hugging Face Hub only, to fetch the pinned `Qwen/Qwen3-Reranker-0.6B` snapshot (~1207 MB in total) at revision `e61197ed4502…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same pins as the repository's pyproject.toml at the generating revision; any `--index-url`/`--find-links` lines are passed to pip as written) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'transformers==4.57.6',
    'huggingface-hub==0.36.2',
    'safetensors==0.8.0',
    'numpy==2.5.3',
]
NOTEBOOK_SOURCE = {
    'repository': 'qwen3-reranker-pipeline',
    'repository_revision': 'a2066cac44b0f165f422cd8cc16107b00f48133a',
    'embedded_module': 'src/qwen3_reranker_pipeline/pipeline.py',
    'embedded_modules': ['src/qwen3_reranker_pipeline/metrics.py', 'src/qwen3_reranker_pipeline/pipeline.py', 'src/qwen3_reranker_pipeline/samples.py'],
    'module_sha256': '12bfcd927e671709e81607c621d5dfa26f5e6a95db956e1dbbea02f36605a0f6',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/qwen3_reranker_pipeline/` @ `a2066cac44b0`)

The next 3 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/3:** `src/qwen3_reranker_pipeline/metrics.py`

In [ ]:
"""Ranking metrics over a query–candidate-list dataset, a random floor and a lexical baseline.

Every query comes with a candidate list — its positive document and its negatives — and a scorer ranks the
list; the rank of the positive gives **recall@1**, **recall@3**, **recall@5** (the positive is within the
first *k*) and **MRR** (mean of 1 / rank). Ties are resolved pessimistically (a tied candidate counts as
ranked above the positive). The **random floor** is what a uniformly random ordering of each query's list
achieves in expectation, averaged over the queries; the **lexical baseline** orders each list by the
Jaccard overlap of lower-cased alphanumeric tokens with the query — what a system with no model gets
from shared words.
"""

from __future__ import annotations

import hashlib
import re
from collections.abc import Mapping, Sequence
from typing import Any

RECALL_AT = (1, 3, 5)
METRIC_DEFINITIONS = {
    "recall@k": (
        "fraction of queries whose positive document is ranked within the first k of the query's candidate "
        "list; ties count against the positive"
    ),
    "mrr": "mean over queries of 1 / rank of the positive document within its candidate list",
    "candidates": "each query's list is its positive followed by its negatives; lists may differ in length",
}
_TOKEN_RE = re.compile(r"[a-z0-9]+")


def record_seed(record_id: str, seed: int) -> int:
    """A stable per-record seed so seeded choices depend only on the record and the base seed."""
    digest = hashlib.sha256(f"{seed}:{record_id}".encode()).digest()
    return int.from_bytes(digest[:8], "big")


def rank_of_positive(scores: Sequence[float], positive_index: int = 0) -> int:
    """1-based rank of `positive_index` under descending `scores`; ties are ranked above the positive."""
    if not 0 <= positive_index < len(scores):
        raise ValueError("positive_index is outside the candidate list")
    target = float(scores[positive_index])
    return 1 + sum(1 for i, s in enumerate(scores) if i != positive_index and float(s) >= target)


def ranking_metrics(ranks: Sequence[int], list_sizes: Sequence[int]) -> dict[str, Any]:
    """Aggregate 1-based ranks of each query's positive (with its list size) into recall@k and MRR."""
    if not ranks:
        raise ValueError("no queries to score")
    if len(ranks) != len(list_sizes):
        raise ValueError("ranks and list_sizes must align")
    for rank, size in zip(ranks, list_sizes, strict=True):
        if not isinstance(size, int) or size < 2:
            raise ValueError("every candidate list needs at least two entries")
        if not isinstance(rank, int) or not 1 <= rank <= size:
            raise ValueError("every rank must be an int in 1..list size")
    out: dict[str, Any] = {
        "n_queries": len(ranks),
        "candidates": {
            "min": min(list_sizes),
            "max": max(list_sizes),
            "mean": sum(list_sizes) / len(list_sizes),
        },
    }
    for k in RECALL_AT:
        out[f"recall@{k}"] = sum(1 for r in ranks if r <= k) / len(ranks)
    out["mrr"] = sum(1.0 / r for r in ranks) / len(ranks)
    out["median_rank"] = sorted(ranks)[len(ranks) // 2]
    out["definitions"] = dict(METRIC_DEFINITIONS)
    return out


def random_floor(list_sizes: Sequence[int]) -> dict[str, Any]:
    """Expected metrics of a uniformly random ordering of every query's candidate list."""
    if not list_sizes or any(not isinstance(n, int) or n < 2 for n in list_sizes):
        raise ValueError("every candidate list needs at least two entries")
    out: dict[str, Any] = {"n_queries": len(list_sizes)}
    for k in RECALL_AT:
        out[f"recall@{k}"] = sum(min(k, n) / n for n in list_sizes) / len(list_sizes)
    out["mrr"] = sum(sum(1.0 / r for r in range(1, n + 1)) / n for n in list_sizes) / len(list_sizes)
    out["baseline"] = "uniformly random ordering of each candidate list (expected values)"
    return out


def _tokens(text: str) -> set[str]:
    return set(_TOKEN_RE.findall(text.lower()))


def jaccard(a: str, b: str) -> float:
    x, y = _tokens(a), _tokens(b)
    if not x or not y:
        return 0.0
    return len(x & y) / len(x | y)


def candidate_list(record: Mapping[str, Any]) -> list[str]:
    """A record's candidate list: its positive first, then its negatives."""
    return [str(record["positive"]), *(str(n) for n in record["negatives"])]


def lexical_baseline(records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    """Order every candidate list by Jaccard token overlap with the query — the no-model floor."""
    ranks, sizes = [], []
    for record in records:
        candidates = candidate_list(record)
        scores = [jaccard(record["query"], doc) for doc in candidates]
        ranks.append(rank_of_positive(scores, 0))
        sizes.append(len(candidates))
    result = ranking_metrics(ranks, sizes)
    result["baseline"] = "Jaccard overlap of lower-cased alphanumeric tokens between query and candidate"
    return result

**Module 2/3:** `src/qwen3_reranker_pipeline/pipeline.py` (carried verbatim; see the note above)

In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import time
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import numpy as np

MODEL_ID = "Qwen/Qwen3-Reranker-0.6B"
MODEL_REVISION = "e61197ed45024b0ed8a2d74b80b4d909f1255473"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "qwen3-reranker-0.6b"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

# Prompt contract from the pinned upstream README ("Using Transformers"): a fixed system prompt, the
# instruction/query/document block, and the assistant prefix with an empty <think> block; the score is
# the softmax over the "no"/"yes" logits at the last position. Token ids are pinned by the snapshot's
# 1_LogitScore/config.json and cross-checked against the tokenizer at load time.
PREFIX = (
    "<|im_start|>system\nJudge whether the Document meets the requirements based on the Query and the "
    'Instruct provided. Note that the answer can only be "yes" or "no".<|im_end|>\n<|im_start|>user\n'
)
SUFFIX = "<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"
DEFAULT_INSTRUCTION = "Given a web search query, retrieve relevant passages that answer the query"
YES_TOKEN, NO_TOKEN = "yes", "no"
YES_TOKEN_ID, NO_TOKEN_ID = 9693, 2152
MAX_TEXT_TOKENS = 8192  # total prompt length incl. prefix/suffix; the pair is truncated longest-first to fit
MAX_TEXT_CHARS = 100_000  # per query or document, pre-tokenisation guard
MAX_PAIRS = 32  # (query, document) pairs per rerank() call
SCORE_KIND = "relevance score, not a calibrated probability"
WEIGHT_FILE = "model.safetensors"
WEIGHT_SHA256 = (
    "27cd75a405b9c1b46b59abfd88aaa209e6fed2a1972cde9b70e7659537c5e65b"  # manifest digest of WEIGHT_FILE
)
PARAMETER_COUNT = 595_776_512  # Qwen3ForCausalLM with the output projection tied to the token embeddings
DECODER_LAYERS = 28  # config.json num_hidden_layers
DEFAULT_TRAINABLE_LAYERS = 2  # the last two decoder layers (31,461,888 parameters)
MAX_TRAIN_TOKENS = 192  # training-only prompt ceiling (inference truncates the pair to MAX_TEXT_TOKENS)
MAX_TRAIN_CANDIDATES = 4  # per query: the positive plus the first negatives, scored together in one list
MAX_EVAL_RECORDS = 2_000
MIN_SCORED_RECORDS = 50  # below this a scored dataset is labelled a small sample
ARTIFACT_FORMAT = "org.valcorza.qwen3-reranker-0.6b.adapter.v1"
ARTIFACT_FORMAT_VERSION = "1.0"
ARTIFACT_WEIGHTS_NAME = "adapter.safetensors"
ARTIFACT_MANIFEST_NAME = "manifest.json"


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check the local snapshot against its DIMER manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = hashlib.sha256()
        with open(file_path, "rb") as fh:
            for chunk in iter(lambda: fh.read(1 << 20), b""):
                digest.update(chunk)
        if digest.hexdigest() != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest.hexdigest()} != manifest {entry['sha256']}")
    return manifest


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def format_pair(query: str, document: str, instruction: str = DEFAULT_INSTRUCTION) -> str:
    """Upstream `format_instruction`: the user-turn body between PREFIX and SUFFIX."""
    return f"<Instruct>: {instruction}\n<Query>: {query}\n<Document>: {document}"


INPUT_SCHEMA: dict[str, Any] = {
    "input": "sequence of (query, document) pairs of two non-empty str; one score is returned per pair",
    "pairs": [1, MAX_PAIRS],
    "text_chars": [1, MAX_TEXT_CHARS],
    "prompt_tokens": [1, MAX_TEXT_TOKENS],
    "score_range": [0.0, 1.0],
    "preprocessing": (
        "each pair becomes '<Instruct>: <instruction>\\n<Query>: …\\n<Document>: …' between the fixed "
        "upstream PREFIX and SUFFIX, truncated longest-first to fit MAX_TEXT_TOKENS; the score is the "
        "two-way softmax share of the yes logit against the no logit at the last position"
    ),
}


def _check_inputs(pairs: Any, instruction: str) -> list[tuple[str, str]]:
    """Raise TypeError/ValueError naming the first violated ceiling; return the pairs as a list."""
    if isinstance(pairs, str | bytes) or not isinstance(pairs, Sequence):
        raise TypeError("pairs must be a list of (query, document) pairs")
    if not 1 <= len(pairs) <= MAX_PAIRS:
        raise ValueError(f"pairs must hold 1..{MAX_PAIRS} items, got {len(pairs)}")
    for i, pair in enumerate(pairs):
        if isinstance(pair, str | bytes) or not isinstance(pair, Sequence) or len(pair) != 2:
            raise TypeError(f"pairs[{i}] must be a (query, document) pair of two str")
        for name, text in zip(("query", "document"), pair, strict=True):
            if not isinstance(text, str):
                raise TypeError(f"pairs[{i}] {name} must be str, got {type(text).__name__}")
            if not text.strip():
                raise ValueError(f"pairs[{i}] {name} is empty")
            if len(text) > MAX_TEXT_CHARS:
                raise ValueError(f"pairs[{i}] {name} has {len(text)} chars; ceiling is {MAX_TEXT_CHARS}")
    if not isinstance(instruction, str) or not instruction.strip():
        raise ValueError("instruction must be a non-empty str")
    return [(pair[0], pair[1]) for pair in pairs]


def validate_inputs(
    pairs: Sequence[Sequence[str]],
    instruction: str = DEFAULT_INSTRUCTION,
    *,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, per-pair observations, verdict).

    Rejection is reported by raising exactly as ``rerank`` would — both route through
    ``_check_inputs``. Prompt-level truncation cannot be observed here because it happens inside
    the tokenizer; ``rerank`` reports it in ``truncated``.
    """
    checked = _check_inputs(pairs, instruction)
    if names is not None and len(names) != len(checked):
        raise ValueError("names must have one entry per pair")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [
            {
                "id": names[i] if names else f"pair-{i}",
                "query_chars": len(query),
                "document_chars": len(document),
            }
            for i, (query, document) in enumerate(checked)
        ],
        "n_pairs": len(checked),
        "instruction": instruction,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    result: Mapping[str, Any], judgements: Sequence[Any] | None = None, *, sample_kind: str = "synthetic"
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even though no metric exists here.

    The repository ships no ranking-metric helper, so the verdict is always ``not-measurable``
    (EVAL9), including when ``judgements`` is supplied: the parameter exists for interface parity
    with the fleet's other pipelines and is recorded in ``reason`` rather than scored. Inventing
    nDCG or MRR here would hide the fact that a real evaluation needs judged candidates over many
    queries and the caller's own metric code.
    """
    scores = result["scores"]
    supplied = judgements is not None
    return {
        "task": "pointwise query-document relevance reranking",
        "score_semantics": (
            f"{SCORE_KIND}: the softmax share of the yes logit against the no logit, in [0, 1]; it "
            "orders candidates for one query, is not comparable as an absolute value across queries "
            "or instructions, and carries no shipped acceptance threshold"
        ),
        "sample_kind": sample_kind,
        "n_pairs": len(scores),
        "metrics": [],
        "baselines": [],
        "verdict": "not-measurable",
        "reason": (
            "ranking quality needs relevance judgements and the repository ships no metric helper"
            + (
                "; judgements were supplied but no metric helper exists to score them here"
                if supplied
                else "; the evaluated sample carries none"
            )
        ),
        "needs": (
            "per-query relevance judgements (binary or graded) over enough queries to state a "
            "dispersion, scored with the caller's own nDCG@k, MRR or precision@k code; a single "
            "query's ordering is a plumbing check, not a retrieval measurement"
        ),
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


@dataclass
class Qwen3RerankerPipeline:
    """Pointwise reranker. `_runner` maps pair bodies to ([no, yes] last-position logits, token counts)."""

    _runner: Callable[[list[str]], tuple[np.ndarray, list[int]]]
    device: str
    adapter: dict[str, Any] | None = field(default=None, repr=False)
    _model: Any = field(default=None, repr=False)
    _tokenizer: Any = field(default=None, repr=False)

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> Qwen3RerankerPipeline:
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            source, kwargs = str(root), dict(local_files_only=True)
        elif allow_download:
            source, kwargs = MODEL_ID, dict(revision=MODEL_REVISION)
        else:
            raise FileNotFoundError(f"no verified snapshot at {root} and allow_download=False")
        # Refuse invalid snapshots before importing model libraries.
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        dtype = torch.bfloat16 if resolved_device.startswith("cuda") else torch.float32
        tokenizer = AutoTokenizer.from_pretrained(
            source, padding_side="left", trust_remote_code=False, **kwargs
        )
        ids = (tokenizer.convert_tokens_to_ids(YES_TOKEN), tokenizer.convert_tokens_to_ids(NO_TOKEN))
        if ids != (YES_TOKEN_ID, NO_TOKEN_ID):
            raise RuntimeError(f"tokenizer maps yes/no to {ids}, expected {(YES_TOKEN_ID, NO_TOKEN_ID)}")
        model = AutoModelForCausalLM.from_pretrained(source, dtype=dtype, trust_remote_code=False, **kwargs)
        model = model.to(resolved_device).eval()
        prefix_ids = tokenizer.encode(PREFIX, add_special_tokens=False)
        suffix_ids = tokenizer.encode(SUFFIX, add_special_tokens=False)
        body_budget = MAX_TEXT_TOKENS - len(prefix_ids) - len(suffix_ids)

        def runner(bodies: list[str]) -> tuple[np.ndarray, list[int]]:
            enc = tokenizer(
                bodies,
                padding=False,
                truncation="longest_first",
                return_attention_mask=False,
                max_length=body_budget,
            )
            enc["input_ids"] = [prefix_ids + row + suffix_ids for row in enc["input_ids"]]
            batch = tokenizer.pad(enc, padding=True, return_tensors="pt")
            batch = batch.to(resolved_device)
            with torch.inference_mode():
                last = model(**batch).logits[:, -1, :]
            pair_logits = torch.stack([last[:, NO_TOKEN_ID], last[:, YES_TOKEN_ID]], dim=1)
            counts = batch["attention_mask"].sum(dim=1).tolist()
            return pair_logits.float().cpu().numpy(), [int(c) for c in counts]

        return cls(runner, resolved_device, _model=model, _tokenizer=tokenizer)

    def _validate(self, pairs: Any, instruction: str) -> list[tuple[str, str]]:
        return _check_inputs(pairs, instruction)

    def rerank(
        self,
        pairs: Sequence[Sequence[str]],
        instruction: str = DEFAULT_INSTRUCTION,
    ) -> dict[str, Any]:
        """Score up to MAX_PAIRS (query, document) pairs; `scores` align with `pairs`; no threshold."""
        pairs = self._validate(pairs, instruction)
        bodies = [format_pair(q, d, instruction) for q, d in pairs]
        logits, n_tokens = self._runner(bodies)
        logits = np.asarray(logits, dtype=np.float64)
        if logits.shape != (len(pairs), 2):
            raise RuntimeError(f"backend returned {logits.shape}, expected ({len(pairs)}, 2)")
        shifted = logits - logits.max(axis=1, keepdims=True)
        probs = np.exp(shifted) / np.exp(shifted).sum(axis=1, keepdims=True)
        scores = probs[:, 1]
        return {
            "scores": [float(s) for s in scores],
            "ranking": [int(i) for i in np.argsort(-scores, kind="stable")],
            "score_kind": SCORE_KIND,
            "instruction": instruction,
            "n_tokens": list(n_tokens),
            "truncated": [n >= MAX_TEXT_TOKENS for n in n_tokens],
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    # ---- adaptation -----------------------------------------------------------------------------------

    def _require_model(self) -> tuple[Any, Any]:
        if self._model is None or self._tokenizer is None:
            raise ValueError(
                "this operation needs a pipeline built with from_pretrained() or from_artifact()"
            )
        return self._model, self._tokenizer

    def _score_pairs(self, pairs: Sequence[tuple[str, str]], instruction: str) -> list[float]:
        """Score any number of pairs through the public contract, MAX_PAIRS at a time."""
        scores: list[float] = []
        for start in range(0, len(pairs), MAX_PAIRS):
            scores.extend(self.rerank(list(pairs[start : start + MAX_PAIRS]), instruction)["scores"])
        return scores

    def evaluate(
        self, records: Sequence[Mapping[str, Any]], *, instruction: str = DEFAULT_INSTRUCTION
    ) -> dict[str, Any]:
        """Rerank every record's candidate list (its positive followed by its negatives) with `rerank` and
        read the rank of the positive: recall@1 / recall@3 / recall@5 and MRR, ties against the positive."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import candidate_list, rank_of_positive, ranking_metrics` removed — names are kernel globals defined by the carried modules
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        checked = validate_dataset(records, min_records=1, max_records=MAX_EVAL_RECORDS)["records"]
        _check_inputs([("x", "y")], instruction)
        started = time.perf_counter()
        pairs: list[tuple[str, str]] = []
        sizes = []
        for record in checked:
            candidates = candidate_list(record)
            sizes.append(len(candidates))
            pairs.extend((record["query"], doc) for doc in candidates)
        scores = self._score_pairs(pairs, instruction)
        ranks = []
        cursor = 0
        for size in sizes:
            ranks.append(rank_of_positive(scores[cursor : cursor + size], 0))
            cursor += size
        metrics = ranking_metrics(ranks, sizes)
        metrics.update(
            {
                "instruction": instruction,
                "n_pairs": len(pairs),
                "verdict": "measured" if len(checked) >= MIN_SCORED_RECORDS else "measured-small-sample",
                "adapted": self.adapter is not None,
                "seconds": round(time.perf_counter() - started, 3),
                "model_id": MODEL_ID,
                "model_revision": MODEL_REVISION,
            }
        )
        return metrics

    @staticmethod
    def lexical_baseline(records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
        """The no-model floor: candidate lists ordered by token overlap with the query (see metrics.py)."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import lexical_baseline` removed — names are kernel globals defined by the carried modules
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        return lexical_baseline(
            validate_dataset(records, min_records=1, max_records=MAX_EVAL_RECORDS)["records"]
        )

    def _trainable_names(self, trainable_layers: int) -> list[str]:
        if not isinstance(trainable_layers, int) or not 1 <= trainable_layers <= DECODER_LAYERS:
            raise ValueError(f"trainable_layers must be an int in 1..{DECODER_LAYERS}")
        model, _ = self._require_model()
        first = DECODER_LAYERS - trainable_layers
        prefixes = tuple(f"model.layers.{k}." for k in range(first, DECODER_LAYERS))
        return [name for name, _p in model.named_parameters() if name.startswith(prefixes)]

    def adapt(
        self,
        train: Sequence[Mapping[str, Any]],
        val: Sequence[Mapping[str, Any]] | None = None,
        *,
        instruction: str = DEFAULT_INSTRUCTION,
        epochs: int = 2,
        lr: float = 2e-5,
        batch_size: int = 4,
        trainable_layers: int = DEFAULT_TRAINABLE_LAYERS,
        train_candidates: int = MAX_TRAIN_CANDIDATES,
        seed: int = 0,
        progress: Callable[[dict[str, Any]], None] | None = None,
    ) -> dict[str, Any]:
        """Bounded listwise fine-tuning on validated query / positive / negatives records.

        Only the last `trainable_layers` decoder layers train (2 by default; the token embeddings — which the
        output projection shares — the earlier layers and the final norm stay frozen). Each query contributes
        one list: its positive and its first `train_candidates - 1` negatives, formatted exactly as `rerank`
        formats them; the relevance logit of every candidate is the yes-minus-no logit at the last position,
        and the loss is the cross-entropy of the positive over its list (listwise softmax). `batch_size`
        counts queries per step; AdamW at a fixed learning rate, gradient clipping at 1.0, seeded shuffling,
        no scheduler; prompts are truncated to MAX_TRAIN_TOKENS **during training only**. Epoch 0 records the
        frozen model's validation ranking metrics; the epoch with the highest validation MRR is kept."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import candidate_list` removed — names are kernel globals defined by the carried modules
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        if not isinstance(epochs, int) or not 1 <= epochs <= 20:
            raise ValueError("epochs must be an int in 1..20")
        if not (0.0 < lr <= 1e-3):
            raise ValueError("lr must be in (0, 1e-3]")
        if not isinstance(batch_size, int) or not 1 <= batch_size <= 16:
            raise ValueError("batch_size must be an int in 1..16")
        if not isinstance(train_candidates, int) or not 2 <= train_candidates <= 8:
            raise ValueError("train_candidates must be an int in 2..8")
        _check_inputs([("x", "y")], instruction)
        names = self._trainable_names(trainable_layers)
        train_checked = validate_dataset(train)["records"]
        val_checked = (
            validate_dataset(val, min_records=1, max_records=MAX_EVAL_RECORDS)["records"] if val else []
        )
        import torch

        torch.manual_seed(seed)
        model, tokenizer = self._require_model()
        started = time.perf_counter()
        wanted = set(names)
        for name, param in model.named_parameters():
            param.requires_grad_(name in wanted)
        params = [p for p in model.parameters() if p.requires_grad]
        n_trainable = sum(p.numel() for p in params)
        optimiser = torch.optim.AdamW(params, lr=lr, weight_decay=0.01)
        device = torch.device(self.device)
        prefix_ids = tokenizer.encode(PREFIX, add_special_tokens=False)
        suffix_ids = tokenizer.encode(SUFFIX, add_special_tokens=False)
        body_budget = MAX_TRAIN_TOKENS - len(prefix_ids) - len(suffix_ids)

        def score_val() -> dict[str, Any] | None:
            if not val_checked:
                return None
            model.eval()
            keep = ("recall@1", "recall@3", "recall@5", "mrr", "n_pairs")
            return {k: v for k, v in self.evaluate(val_checked, instruction=instruction).items() if k in keep}

        def relevance(bodies: list[str]) -> torch.Tensor:
            enc = tokenizer(
                bodies,
                padding=False,
                truncation="longest_first",
                max_length=body_budget,
                return_attention_mask=False,
            )
            enc["input_ids"] = [prefix_ids + row + suffix_ids for row in enc["input_ids"]]
            batch = tokenizer.pad(enc, padding=True, return_tensors="pt").to(device)
            last = model(**batch).logits[:, -1, :].float()
            return last[:, YES_TOKEN_ID] - last[:, NO_TOKEN_ID]

        history: list[dict[str, Any]] = []
        entry: dict[str, Any] = {"epoch": 0, "train_loss": None, "val": score_val(), "note": "frozen model"}
        history.append(entry)
        if progress:
            progress(entry)
        best_mrr = entry["val"]["mrr"] if entry["val"] else -math.inf
        best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in wanted}
        best_epoch = 0
        generator = torch.Generator().manual_seed(seed)
        lists = [candidate_list(r)[:train_candidates] for r in train_checked]
        for epoch in range(1, epochs + 1):
            model.train()
            order = torch.randperm(len(train_checked), generator=generator).tolist()
            losses = []
            for start in range(0, len(order), batch_size):
                chosen = order[start : start + batch_size]
                bodies, spans = [], []
                for i in chosen:
                    spans.append((len(bodies), len(lists[i])))
                    bodies.extend(
                        format_pair(train_checked[i]["query"], doc, instruction) for doc in lists[i]
                    )
                logits = relevance(bodies)
                loss = torch.stack(
                    [
                        torch.nn.functional.cross_entropy(
                            logits[a : a + n][None], torch.zeros(1, dtype=torch.long, device=device)
                        )
                        for a, n in spans
                    ]
                ).mean()
                optimiser.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                optimiser.step()
                losses.append(float(loss.detach()))
            model.eval()
            entry = {"epoch": epoch, "train_loss": sum(losses) / max(len(losses), 1), "val": score_val()}
            history.append(entry)
            if progress:
                progress(entry)
            current = entry["val"]["mrr"] if entry["val"] else math.inf
            if current > best_mrr or not entry["val"]:
                best_mrr = current
                best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in wanted}
                best_epoch = epoch
        merged = dict(model.state_dict())
        merged.update(best_state)
        model.load_state_dict(merged, strict=True)
        model.eval()
        for param in model.parameters():
            param.requires_grad_(False)
        self.adapter = {
            "objective": "listwise cross-entropy of the positive over its candidates (yes-minus-no logit)",
            "instruction": instruction,
            "trainable_layers": trainable_layers,
            "trainable_names": names,
            "n_trainable": n_trainable,
            "n_total": sum(p.numel() for p in model.parameters()),
            "epochs": epochs,
            "best_epoch": best_epoch,
            "selection": "highest validation MRR" if val_checked else "final epoch (no validation split)",
            "lr": lr,
            "batch_size": batch_size,
            "train_candidates": train_candidates,
            "max_train_tokens": MAX_TRAIN_TOKENS,
            "n_train": len(train_checked),
            "n_train_pairs": sum(len(lst) for lst in lists),
            "n_val": len(val_checked),
            "seed": seed,
            "history": history,
            "seconds": round(time.perf_counter() - started, 2),
        }
        return dict(self.adapter)

    # ---- artifacts ------------------------------------------------------------------------------------

    def save_artifact(self, output_dir: str | Path, metadata: Mapping[str, Any] | None = None) -> Path:
        """Write the adapted decoder-layer tensors as safetensors with a manifest naming the base."""
        if self.adapter is None:
            raise ValueError("nothing to save: call adapt() first")
        model, _ = self._require_model()
        from safetensors.torch import save_file

        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        names = set(self.adapter["trainable_names"])
        tensors = {k: v.detach().cpu().contiguous() for k, v in model.state_dict().items() if k in names}
        weights_path = out / ARTIFACT_WEIGHTS_NAME
        save_file(tensors, str(weights_path), metadata={"format": "pt"})
        manifest = {
            "format": ARTIFACT_FORMAT,
            "format_version": ARTIFACT_FORMAT_VERSION,
            "base_model": {
                "id": MODEL_ID,
                "revision": MODEL_REVISION,
                "key": MODEL_KEY,
                "weight_file": WEIGHT_FILE,
                "weight_sha256": WEIGHT_SHA256,
            },
            "adapter": {k: v for k, v in self.adapter.items() if k not in ("history", "trainable_names")},
            "history": self.adapter["history"],
            "tensors": sorted(tensors),
            "files": [
                {
                    "path": ARTIFACT_WEIGHTS_NAME,
                    "bytes": weights_path.stat().st_size,
                    "sha256": _sha256(weights_path),
                }
            ],
            "metadata": dict(metadata or {}),
        }
        (out / ARTIFACT_MANIFEST_NAME).write_text(
            json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"
        )
        return out

    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:
        """Verify an adapter's manifest and digest, then overwrite exactly the tensors it carries."""
        root = Path(artifact_dir)
        manifest = json.loads((root / ARTIFACT_MANIFEST_NAME).read_text(encoding="utf-8"))
        if manifest.get("format") != ARTIFACT_FORMAT:
            raise ValueError(f"artifact format {manifest.get('format')!r} != {ARTIFACT_FORMAT!r}")
        base = manifest.get("base_model", {})
        if (base.get("id"), base.get("revision"), base.get("weight_sha256")) != (
            MODEL_ID,
            MODEL_REVISION,
            WEIGHT_SHA256,
        ):
            raise ValueError("artifact was adapted from a different base model, revision or weight file")
        entry = manifest["files"][0]
        weights_path = root / entry["path"]
        if not weights_path.is_file():
            raise FileNotFoundError(f"artifact weights missing: {weights_path}")
        if _sha256(weights_path) != entry["sha256"] or weights_path.stat().st_size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: digest or size mismatch; refusing to load")
        model, _ = self._require_model()
        from safetensors.torch import load_file

        tensors = load_file(str(weights_path))
        if sorted(tensors) != manifest["tensors"]:
            raise ValueError("artifact tensor names differ from its manifest")
        state = model.state_dict()
        for key, value in tensors.items():
            if key not in state or not key.startswith("model.layers."):
                raise ValueError(
                    f"artifact tensor {key} is not an adaptable decoder-layer tensor of the base"
                )
            if tuple(value.shape) != tuple(state[key].shape):
                raise ValueError(
                    f"artifact tensor {key}: shape {tuple(value.shape)} != {tuple(state[key].shape)}"
                )
        merged = dict(state)
        merged.update({k: v.to(state[k].dtype) for k, v in tensors.items()})
        model.load_state_dict(merged, strict=True)
        model.eval()
        self.adapter = {
            **manifest["adapter"],
            "trainable_names": manifest["tensors"],
            "history": manifest.get("history", []),
        }
        return manifest

    @classmethod
    def from_artifact(
        cls,
        artifact_dir: str | Path,
        *,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> Qwen3RerankerPipeline:
        pipeline = cls.from_pretrained(device=device, weights_dir=weights_dir, allow_download=allow_download)
        pipeline.load_artifact(artifact_dir)
        return pipeline

**Module 3/3:** `src/qwen3_reranker_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
"""Query / positive / negatives dataset contract for reranker fine-tuning: the pinned Banking77 sample,
validation, seeded splitting, BYOD loaders and CSV export.

The default dataset is **real** and a reranking task the model was not tuned for: Banking77 (Casanueva et
al., 2020; CC BY 4.0), 13,083 customer-support messages labelled with 77 fine-grained banking intents. Two
CSV files (`train.csv`, `test.csv`) are fetched from the PolyAI `task-specific-datasets` repository at a
pinned commit and refused on any byte-size or SHA-256 mismatch. Every intent name becomes a short
**document** (`card_arrival` → `card arrival`); each message is a **query** whose positive document is its
intent phrase, and its **negatives** are a seeded shortlist of other intent phrases — the hardest few by
token overlap with the message plus random ones — so reranking the shortlist is intent detection over a
candidate list. Training and validation queries are drawn from `train.csv`, test queries from `test.csv` —
the release's own partition — balanced over the 77 intents.

A record is ``{id, query, positive, negatives}``; its candidate list is the positive then the negatives.
"""

from __future__ import annotations

import csv
import hashlib
import io
import json
import random
import re
import urllib.request
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Any

# standalone rewrite (build_notebook.py): `from .metrics import jaccard, record_seed` removed — names are kernel globals defined by the carried modules
# standalone rewrite (build_notebook.py): `from .pipeline import MAX_TEXT_CHARS, MODEL_ID` removed — names are kernel globals defined by the carried modules

CORPUS_NAME = "Banking77 (messages → intent-phrase shortlists)"
CORPUS_RELEASE = "PolyAI-LDN/task-specific-datasets @ 57ec275d8078af65b7731c2a98be812d844a6d6b"
CORPUS_BASE_URL = (
    "https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/"
    "57ec275d8078af65b7731c2a98be812d844a6d6b/banking_data/"
)
CORPUS_FILES = {
    "train": ("train.csv", 839_073, "b06e26ac675513959a63135f11b94ea7786ed02da65db93a5650d8838cbc664b"),
    "test": ("test.csv", 239_961, "d12d6e3bc4c3103966ae786dc435913c0c563dfa328f5a3646d0e62cfeeb474d"),
}
CORPUS_LICENSE = "CC BY 4.0 (Casanueva et al. 2020; PolyAI-LDN/task-specific-datasets)"
CORPUS_ROWS = {"train": 10_003, "test": 3_080}
CORPUS_INTENTS = 77
DEFAULT_CACHE_DIR = Path("weights") / "banking77"
TASK_INSTRUCTION = (
    "Given a customer support message, judge whether the document names the banking intent it expresses"
)
SAMPLE_SEED = 42
SAMPLE_SPLIT = {"train": 231, "validation": 77, "test": 154}  # 3 / 1 / 2 per intent, balanced over 77
SAMPLE_NEGATIVES = 5  # per query: SAMPLE_HARD_NEGATIVES highest-overlap other intents plus seeded random ones
SAMPLE_HARD_NEGATIVES = 3
MAX_NEGATIVES = 15
NEGATIVE_SEPARATOR = " | "  # the CSV column format of `negatives`
MIN_RECORDS = 8
MAX_RECORDS = 20_000
MAX_DOCUMENT_CHARS = 1_000
_ID_RE = re.compile(r"^[A-Za-z0-9_.:-]{1,64}$")


def _sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def intent_phrase(intent: str) -> str:
    """The document text of an intent: its snake_case name as words (`card_arrival` → `card arrival`)."""
    return " ".join(intent.strip().split("_"))


def fetch_corpus(*, cache_dir: str | Path | None = None, fetcher: Any = None) -> dict[str, bytes]:
    """Return the two pinned Banking77 CSVs (bytes) from the cache or the project repository, verified."""
    cache = Path(cache_dir) if cache_dir is not None else DEFAULT_CACHE_DIR
    cache.mkdir(parents=True, exist_ok=True)
    out = {}
    for split, (name, size, digest) in CORPUS_FILES.items():
        local = cache / name
        data = local.read_bytes() if local.is_file() else b""
        if len(data) != size or _sha256_bytes(data) != digest:
            url = CORPUS_BASE_URL + name
            if fetcher is not None:
                data = fetcher(url)
            else:
                with urllib.request.urlopen(url, timeout=120) as response:  # noqa: S310 (pinned https URL)
                    data = response.read()
            if len(data) != size or _sha256_bytes(data) != digest:
                raise ValueError(
                    f"{name}: fetched {len(data)} bytes with sha256 {_sha256_bytes(data)[:16]}…, "
                    f"pinned {size} / {digest[:16]}…"
                )
            local.write_bytes(data)
        out[split] = data
    return out


def read_corpus(files: Mapping[str, bytes]) -> dict[str, list[dict[str, Any]]]:
    """Parse the CSV members (columns `text`, `category`) into flat records keeping the raw intent name."""
    out = {}
    for split in CORPUS_FILES:
        if split not in files:
            raise ValueError(f"corpus is missing the {split} file")
        rows = list(csv.DictReader(io.StringIO(files[split].decode("utf-8"))))
        if not rows or {"text", "category"} - set(rows[0]):
            raise ValueError(f"{split}: expected columns text and category")
        if len(rows) != CORPUS_ROWS[split]:
            raise ValueError(f"{split}: {len(rows)} rows, expected {CORPUS_ROWS[split]}")
        out[split] = [
            {"id": f"{split}-{i:05d}", "text": r["text"].strip(), "intent": r["category"].strip()}
            for i, r in enumerate(rows)
        ]
        intents = {r["intent"] for r in out[split]}
        if len(intents) != CORPUS_INTENTS:
            raise ValueError(f"{split}: {len(intents)} intents, expected {CORPUS_INTENTS}")
    return out


def filter_records(records: Sequence[Mapping[str, Any]]) -> list[dict[str, Any]]:
    """Turn corpus rows into query–positive pairs; drop empty, over-long and repeated queries."""
    seen: set[str] = set()
    kept = []
    for record in records:
        query = str(record["text"]).strip()
        key = query.lower()
        if not query or key in seen or len(query) > MAX_TEXT_CHARS:
            continue
        seen.add(key)
        intent = str(record["intent"])
        kept.append({"id": record["id"], "query": query, "positive": intent_phrase(intent), "intent": intent})
    return kept


def sample_negatives(
    query: str,
    positive: str,
    phrases: Sequence[str],
    *,
    seed: int,
    n_negatives: int = SAMPLE_NEGATIVES,
    n_hard: int = SAMPLE_HARD_NEGATIVES,
) -> list[str]:
    """A seeded shortlist of other phrases: the `n_hard` with the highest token overlap with the query (ties
    broken by phrase order), then random others up to `n_negatives`."""
    others = [p for p in phrases if p != positive]
    if len(others) < n_negatives:
        raise ValueError(f"need {n_negatives} other phrases, have {len(others)}")
    hard = sorted(others, key=lambda p: (-jaccard(query, p), p))[:n_hard]
    rest = [p for p in others if p not in hard]
    random.Random(seed).shuffle(rest)
    return hard + rest[: n_negatives - len(hard)]


def build_sample_dataset(
    corpus: Mapping[str, Sequence[Mapping[str, Any]]],
    *,
    seed: int = SAMPLE_SEED,
    sizes: Mapping[str, int] | None = None,
) -> dict[str, list[dict[str, Any]]]:
    """Balanced seeded draws over all 77 intents with a seeded shortlist of negatives per query: training and
    validation from `train` (disjoint queries), test from `test`."""
    sizes = dict(sizes or SAMPLE_SPLIT)
    for name, size in sizes.items():
        if size % CORPUS_INTENTS:
            raise ValueError(f"{name} size {size} is not a multiple of the {CORPUS_INTENTS} intents")
    rng = random.Random(seed)
    pools = {"train": filter_records(corpus["train"]), "test": filter_records(corpus["test"])}
    intents = sorted({r["intent"] for r in pools["train"]})
    phrases = [intent_phrase(intent) for intent in intents]
    by_intent = {
        split: {intent: [r for r in pool if r["intent"] == intent] for intent in intents}
        for split, pool in pools.items()
    }
    for split in by_intent.values():
        for records in split.values():
            rng.shuffle(records)
    cursor = dict.fromkeys(intents, 0)
    out: dict[str, list[dict[str, Any]]] = {}
    for name, size in sizes.items():
        source = "test" if name == "test" else "train"
        per_intent = size // CORPUS_INTENTS
        picked = []
        for intent in intents:
            pool = by_intent[source][intent]
            start = cursor[intent] if source == "train" else 0
            chunk = pool[start : start + per_intent]
            if len(chunk) < per_intent:
                raise ValueError(
                    f"{name}: only {len(chunk)} records available for {intent!r}, need {per_intent}"
                )
            picked.extend(chunk)
            if source == "train":
                cursor[intent] = start + per_intent
        rng.shuffle(picked)
        out[name] = []
        for i, r in enumerate(picked):
            rid = f"{name}-{i:04d}"
            out[name].append(
                {
                    "id": rid,
                    "query": r["query"],
                    "positive": r["positive"],
                    "negatives": sample_negatives(
                        r["query"], r["positive"], phrases, seed=record_seed(rid, seed)
                    ),
                    "intent": r["intent"],
                }
            )
    return out


def fetch_sample_dataset(
    *,
    cache_dir: str | Path | None = None,
    fetcher: Any = None,
    seed: int = SAMPLE_SEED,
    sizes: Mapping[str, int] | None = None,
) -> dict[str, list[dict[str, Any]]]:
    """The tutorial splits from the pinned corpus."""
    corpus = read_corpus(fetch_corpus(cache_dir=cache_dir, fetcher=fetcher))
    return build_sample_dataset(corpus, seed=seed, sizes=sizes)


def _check_text(value: Any, label: str, ceiling: int) -> str:
    if not isinstance(value, str):
        raise ValueError(f"{label} must be a string")
    if not value.strip():
        raise ValueError(f"{label} is empty")
    if len(value) > ceiling:
        raise ValueError(f"{label} has {len(value)} chars; ceiling is {ceiling}")
    return value.strip()


def _check_record(record: Any, index: int) -> dict[str, Any]:
    label = f"records[{index}]"
    if not isinstance(record, Mapping):
        raise ValueError(f"{label} must be a mapping with id/query/positive/negatives")
    for key in ("id", "query", "positive", "negatives"):
        if key not in record:
            raise ValueError(f"{label} is missing {key!r}")
    rid = record["id"]
    if not isinstance(rid, str) or not _ID_RE.match(rid):
        raise ValueError(f"{label}: id must match {_ID_RE.pattern}")
    item = {
        "id": rid,
        "query": _check_text(record["query"], f"{label}: query", MAX_TEXT_CHARS),
        "positive": _check_text(record["positive"], f"{label}: positive", MAX_DOCUMENT_CHARS),
    }
    negatives = record["negatives"]
    if isinstance(negatives, str | bytes) or not isinstance(negatives, Sequence):
        raise ValueError(f"{label}: negatives must be a list of 1..{MAX_NEGATIVES} documents")
    if not 1 <= len(negatives) <= MAX_NEGATIVES:
        raise ValueError(f"{label}: negatives must hold 1..{MAX_NEGATIVES} documents, got {len(negatives)}")
    checked = [
        _check_text(n, f"{label}: negatives[{j}]", MAX_DOCUMENT_CHARS) for j, n in enumerate(negatives)
    ]
    if len(set(checked)) != len(checked):
        raise ValueError(f"{label}: negatives repeat a document")
    if item["positive"] in checked:
        raise ValueError(f"{label}: a negative equals the positive")
    item["negatives"] = checked
    if "intent" in record:
        item["intent"] = str(record["intent"])
    return item


def validate_dataset(
    records: Sequence[Mapping[str, Any]], *, min_records: int = MIN_RECORDS, max_records: int = MAX_RECORDS
) -> dict[str, Any]:
    """Structural validation of a query / positive / negatives dataset; raises before any model import."""
    if isinstance(records, Mapping) or not isinstance(records, Sequence) or isinstance(records, (str, bytes)):
        raise ValueError("records must be a list of {id, query, positive, negatives} mappings")
    if not min_records <= len(records) <= max_records:
        raise ValueError(f"{len(records)} records; {min_records}..{max_records} are required")
    checked = []
    ids: set[str] = set()
    queries: set[str] = set()
    for index, record in enumerate(records):
        item = _check_record(record, index)
        if item["id"] in ids:
            raise ValueError(f"duplicate id {item['id']!r}")
        ids.add(item["id"])
        queries.add(item["query"].lower())
        checked.append(item)
    sizes = [1 + len(r["negatives"]) for r in checked]
    return {
        "records": checked,
        "n_records": len(checked),
        "unique_queries": len(queries),
        "n_documents": len(documents(checked)),
        "candidates": {"min": min(sizes), "max": max(sizes), "mean": sum(sizes) / len(sizes)},
        "query_chars": {
            "min": min(len(r["query"]) for r in checked),
            "max": max(len(r["query"]) for r in checked),
        },
        "digest": dataset_digest(checked),
        "model_id": MODEL_ID,
    }


def documents(records: Sequence[Mapping[str, Any]]) -> list[str]:
    """The sorted unique documents (positives and negatives) of a dataset."""
    positives = {str(r["positive"]).strip() for r in records}
    negatives = {str(n).strip() for r in records for n in r["negatives"]}
    return sorted(positives | negatives)


def dataset_digest(records: Sequence[Mapping[str, Any]]) -> str:
    payload = [[r["id"], r["query"], r["positive"], list(r["negatives"])] for r in records]
    return _sha256_bytes(json.dumps(payload, ensure_ascii=False, separators=(",", ":")).encode("utf-8"))


def check_split_disjoint(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:
    """Assert no lower-cased query appears in two splits (leakage check)."""
    seen: dict[str, str] = {}
    for name, records in splits.items():
        for record in records:
            key = str(record["query"]).lower()
            if key in seen and seen[key] != name:
                raise ValueError(f"query {record['query'][:60]!r} appears in both {seen[key]} and {name}")
            seen[key] = name
    return {name: len(records) for name, records in splits.items()}


def split_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    val_fraction: float = 0.15,
    test_fraction: float = 0.2,
    seed: int = 0,
) -> dict[str, list[dict[str, Any]]]:
    """Seeded shuffle of a BYOD dataset into train/validation/test after de-duplicating queries."""
    if not (0.0 <= val_fraction < 1.0 and 0.0 < test_fraction < 1.0 and val_fraction + test_fraction < 1.0):
        raise ValueError("fractions must satisfy 0 <= val < 1, 0 < test < 1, val + test < 1")
    checked = validate_dataset(records)["records"]
    seen: set[str] = set()
    unique = []
    for record in checked:
        key = record["query"].lower()
        if key not in seen:
            seen.add(key)
            unique.append(record)
    random.Random(seed).shuffle(unique)
    n_test = max(1, round(len(unique) * test_fraction))
    n_val = round(len(unique) * val_fraction)
    splits = {
        "test": unique[:n_test],
        "validation": unique[n_test : n_test + n_val],
        "train": unique[n_test + n_val :],
    }
    if len(splits["train"]) < MIN_RECORDS:
        raise ValueError(
            f"split leaves {len(splits['train'])} training records; at least {MIN_RECORDS} are required"
        )
    return splits


def load_byod_dataset(path: str | Path) -> list[dict[str, Any]]:
    """Read `{id, query, positive, negatives}` records from CSV (columns id, query, positive, negatives — the
    negatives separated by ` | `), a JSON array or JSONL (negatives as a list)."""
    file_path = Path(path)
    if not file_path.is_file():
        raise FileNotFoundError(f"dataset not found: {file_path}")
    suffix = file_path.suffix.lower()
    text = file_path.read_text(encoding="utf-8")
    if suffix == ".csv":
        rows = list(csv.DictReader(io.StringIO(text)))
        missing = {"id", "query", "positive", "negatives"} - set(rows[0].keys() if rows else set())
        if missing:
            raise ValueError(f"CSV is missing columns {sorted(missing)}")
        return [
            {
                "id": r["id"],
                "query": r["query"],
                "positive": r["positive"],
                "negatives": [
                    n.strip() for n in r["negatives"].split(NEGATIVE_SEPARATOR.strip()) if n.strip()
                ],
            }
            for r in rows
        ]
    if suffix == ".jsonl":
        return [json.loads(line) for line in text.splitlines() if line.strip()]
    if suffix == ".json":
        data = json.loads(text)
        if not isinstance(data, list):
            raise ValueError("JSON dataset must be an array of records")
        return data
    raise ValueError("BYOD datasets must be .csv, .json or .jsonl")


def write_dataset_csv(records: Sequence[Mapping[str, Any]], path: str | Path) -> Path:
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    with open(out, "w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=["id", "query", "positive", "negatives"])
        writer.writeheader()
        for record in records:
            writer.writerow(
                {
                    "id": record["id"],
                    "query": record["query"],
                    "positive": record["positive"],
                    "negatives": NEGATIVE_SEPARATOR.join(record["negatives"]),
                }
            )
    return out

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `13`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `e61197ed4502…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `Qwen3RerankerPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "qwen3-reranker-0.6b",
  "modelId": "Qwen/Qwen3-Reranker-0.6B",
  "revision": "e61197ed45024b0ed8a2d74b80b4d909f1255473",
  "files": [
    {
      "path": "1_LogitScore/config.json",
      "bytes": 57,
      "sha256": "73e3156450564d8a98b7e47bcf5aace0f29600828b51937da545571e84db3ff3"
    },
    {
      "path": "README.md",
      "bytes": 14742,
      "sha256": "5bba8c734f6dd3ae48317b4139317e45a7fce48fc55e15670b23a0dd15492ab6"
    },
    {
      "path": "chat_template.jinja",
      "bytes": 741,
      "sha256": "6f682162495ec5b39fd9005c01b6aa2a74669379fe967039f1e2cbbe8752369d"
    },
    {
      "path": "config.json",
      "bytes": 727,
      "sha256": "d479c427a9ca5295218063d4f9aca4f297ab4ac27487cca7af42c84643d51ef0"
    },
    {
      "path": "config_sentence_transformers.json",
      "bytes": 325,
      "sha256": "6a153d6696f78fd588c1c728967f0b773ea869d3c6028f151ce71ebe49140762"
    },
    {
      "path": "generation_config.json",
      "bytes": 214,
      "sha256": "81051cd3f6e77013827148d0b8a6ead93f8ac390d5ab805f849199f0af6a08db"
    },
    {
      "path": "merges.txt",
      "bytes": 1671853,
      "sha256": "8831e4f1a044471340f7c0a83d7bd71306a5b867e95fd870f74d0c5308a904d5"
    },
    {
      "path": "model.safetensors",
      "bytes": 1191588280,
      "sha256": "27cd75a405b9c1b46b59abfd88aaa209e6fed2a1972cde9b70e7659537c5e65b"
    },
    {
      "path": "modules.json",
      "bytes": 280,
      "sha256": "6f13b6b4a89e577b591b2077bca40c67c26541a6740a8809267cb474f90806a9"
    },
    {
      "path": "sentence_bert_config.json",
      "bytes": 362,
      "sha256": "3234ebd224d492cbe8d55d5ec80a3f408451c4db3005bafb64fe1c51c763e01e"
    },
    {
      "path": "tokenizer.json",
      "bytes": 11422654,
      "sha256": "aeb13307a71acd8fe81861d94ad54ab689df773318809eed3cbe794b4492dae4"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 9706,
      "sha256": "253153d0738ceb4c668d2eff957714dd2bea0b56de772a9fdccd96cbf517e6a0"
    },
    {
      "path": "vocab.json",
      "bytes": 2776833,
      "sha256": "ca10d7e9fb3ed18575dd1e277a2579c16d108e32f27439684afa0e10b1440910"
    }
  ],
  "totalBytes": 1207486774
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = Qwen3RerankerPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Referenced corpus, validation and split

`fetch_corpus` downloads the two pinned Banking77 CSV files (or reads them from the cache), refuses a byte-size or SHA-256 mismatch per file before it is parsed, and `read_corpus` checks the columns, row counts and the 77 intents of each member. `build_sample_dataset` turns every message into a `{id, query, positive, negatives}` record whose positive is the intent name as words and whose negatives are a seeded shortlist (`sample_negatives`: the three other phrases with the highest token overlap with the message, then two seeded random ones — the seed is a function of the record id, so the shortlist is reproducible), drops repeated messages, and draws 3 training and 1 validation record per intent from the `train` member (disjoint messages) and 2 test records per intent from the `test` member by a seeded shuffle — the release's own partition, balanced over all 77 intents. `validate_dataset` then checks every record against the contract, `check_split_disjoint` asserts no message appears in two splits, and the training split is written to `outputs/qwen3_reranker_train.csv` in the shape BYOD expects. `INSTRUCTION` is the judgement every pair carries in later cells.

Look for: 10,003 + 3,080 raw rows, two digests, splits 231 / 77 / 154, six candidates per query, and four refusal probes — a duplicate id, an empty query, a negative equal to the positive and a dataset too small to split — each rejected before `torch` does anything.

In [ ]:
import hashlib
import io
import json

USE_BYOD = False  # @param {type:"boolean"}
SPLIT_SEED = 42  # @param {type:"integer"}
INSTRUCTION = 'Given a customer support message, judge whether the document names the banking intent it expresses'  # @param {type:"string"}

os.makedirs('outputs', exist_ok=True)
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    file_name, payload = next(iter(uploaded.items()))
    byod_path = Path('work') / file_name
    byod_path.parent.mkdir(parents=True, exist_ok=True)
    byod_path.write_bytes(payload)
    records = load_byod_dataset(byod_path)
    splits = split_dataset(records, seed=SPLIT_SEED)
    data_source = 'BYOD (' + file_name + ')'
    raw_rows = {'byod': len(records)}
else:
    corpus = read_corpus(fetch_corpus(cache_dir='weights/banking77'))
    raw_rows = {name: len(part) for name, part in corpus.items()}
    splits = build_sample_dataset(corpus, seed=SPLIT_SEED)
    data_source = f'{CORPUS_NAME} ({CORPUS_RELEASE}; {CORPUS_LICENSE})'
train_records, val_records, test_records = splits['train'], splits['validation'], splits['test']
dataset_manifests = {name: validate_dataset(part) for name, part in splits.items()}
disjoint = check_split_disjoint(splits)
write_dataset_csv(train_records, 'outputs/qwen3_reranker_train.csv')
print({'data_source': data_source, 'instruction': INSTRUCTION, 'raw_rows': raw_rows, 'splits': disjoint, 'n_documents': len(documents(test_records)), 'file_sha256': {k: v[2][:12] + '...' for k, v in CORPUS_FILES.items()}})
for name, manifest in dataset_manifests.items():
    print({name: {'n': manifest['n_records'], 'unique_queries': manifest['unique_queries'], 'candidates': manifest['candidates'], 'query_chars': manifest['query_chars'], 'digest': manifest['digest'][:16] + '...'}})
print({'example': {k: train_records[0][k] for k in ('id', 'query', 'positive', 'negatives')}})

probes = {
    'duplicate id': [{**r, 'id': 'same'} for r in train_records[:8]],
    'empty query': [{**train_records[0], 'query': '   '}, *train_records[1:8]],
    'negative equals positive': [{**train_records[0], 'negatives': [train_records[0]['positive']]}, *train_records[1:8]],
    'too small': train_records[:3],
}
for name, probe in probes.items():
    try:
        validate_dataset(probe)
        print({'probe': name, 'verdict': 'accepted'})
    except (TypeError, ValueError) as exc:
        print({'probe': name, 'rejected': str(exc)[:110]})

## 5. Rerank one shortlist through the inference contract

Before any adaptation, the reranking contract is exercised as it always was, on the first test record's shortlist. `validate_inputs` applies exactly the checks `rerank` applies — both route through the same private `_check_inputs` — so pair shape, batch size 1..`MAX_PAIRS`, non-empty texts, the character ceiling and a non-empty instruction are enforced identically; it returns an input manifest, and an empty-document pair is validated too and its rejection recorded as a finding. `rerank` returns one `score` per pair (the `yes` share of a two-way softmax — a **relevance score, not a calibrated probability**), the `ranking` as a permutation of the pairs, `n_tokens` and `truncated` flags. The ranked shortlist is printed with the gold intent marked; the frozen rankings of three test shortlists are kept as the *before* column for Section 9.

In [ ]:
import time

probe_records = test_records[:3]
record = probe_records[0]
candidates = candidate_list(record)
pairs = [(record['query'], doc) for doc in candidates]
print({'ceilings': {'MAX_PAIRS': MAX_PAIRS, 'MAX_TEXT_CHARS': MAX_TEXT_CHARS, 'MAX_TEXT_TOKENS': MAX_TEXT_TOKENS, 'MAX_TRAIN_TOKENS': MAX_TRAIN_TOKENS, 'MAX_TRAIN_CANDIDATES': MAX_TRAIN_CANDIDATES}, 'yes_no_token_ids': (YES_TOKEN_ID, NO_TOKEN_ID)})
input_manifest = validate_inputs(pairs, INSTRUCTION, names=[f'{record["id"]}-cand-{i}' for i in range(len(pairs))])
try:
    validate_inputs([(record['query'], '   ')], INSTRUCTION)
except ValueError as exc:
    input_manifest['findings'].append({'input': 'empty-document-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/qwen3_reranker_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
started = time.perf_counter()
result = pipe.rerank(pairs, instruction=INSTRUCTION)
rerank_seconds = round(time.perf_counter() - started, 3)
scores = result['scores']
checks = {
    'one_score_per_pair': len(scores) == len(pairs),
    'scores_in_unit_interval': all(0.0 <= s <= 1.0 for s in scores),
    'ranking_is_permutation': sorted(result['ranking']) == list(range(len(pairs))),
    'nothing_truncated': not any(result['truncated']),
    'instruction_echoed': result['instruction'] == INSTRUCTION and result['score_kind'] == SCORE_KIND,
}
if not all(checks.values()):
    raise RuntimeError(f'rerank output failed a sanity check: {checks}')
print({'query': record['query'], 'gold': record['positive'], 'seconds': rerank_seconds, 'n_tokens': result['n_tokens'], 'checks': checks, 'findings': len(input_manifest['findings']), 'score_semantics': result['score_kind'] + '; no threshold shipped'})
for rank, index in enumerate(result['ranking'], start=1):
    print(f"{rank}. {'*' if index == 0 else ' '} {candidates[index]:<40} score {scores[index]:.4f}")

def ranked_shortlist(rec):
    docs = candidate_list(rec)
    out = pipe.rerank([(rec['query'], d) for d in docs], instruction=INSTRUCTION)
    return [(docs[i], round(out['scores'][i], 4)) for i in out['ranking']]

before = {r['id']: ranked_shortlist(r) for r in probe_records}
print({'shortlists_ranked_by_the_frozen_model': len(before)})

## 6. The random floor, the lexical baseline and the frozen model on the test split

Three numbers frame the adaptation, all over the same 154 six-entry shortlists. The **random floor** is what a uniformly random ordering achieves in expectation (recall@1 = 1 / 6, MRR ≈ 0.41). The **lexical baseline** orders each shortlist by Jaccard token overlap with the message — and because three of the five negatives were chosen for high overlap, this floor is deliberately hard to beat by words alone. `pipe.evaluate` scores every query–candidate pair through `rerank` (`MAX_PAIRS` at a time), orders each shortlist by the relevance score and reads the rank of the positive; ties are counted against it. Look for the frozen reranker well above both — the build record saw recall@1 around 73.4 % and MRR around 0.843 — and read `median_rank` beside the means. About 132 s on CPU: 924 forward passes.

In [ ]:
def brief(m):
    return {k: round(m[k], 4) for k in ('recall@1', 'recall@3', 'recall@5', 'mrr')} | {'median_rank': m.get('median_rank')}

floor = random_floor([1 + len(r['negatives']) for r in test_records])
print({'random_floor': {k: round(floor[k], 4) for k in ('recall@1', 'recall@3', 'recall@5', 'mrr')}, 'baseline': floor['baseline']})
t0 = time.perf_counter()
baseline_lexical = pipe.lexical_baseline(test_records)
print({'lexical_baseline': brief(baseline_lexical), 'baseline': baseline_lexical['baseline'], 'seconds': round(time.perf_counter() - t0, 1)})
t0 = time.perf_counter()
frozen_test = pipe.evaluate(test_records, instruction=INSTRUCTION)
print({'frozen_model_test': brief(frozen_test), 'n_queries': frozen_test['n_queries'], 'n_pairs': frozen_test['n_pairs'], 'verdict': frozen_test['verdict'], 'adapted': frozen_test['adapted'], 'seconds': round(time.perf_counter() - t0, 1)})
print({'definitions': frozen_test['definitions']})
assert frozen_test['n_queries'] == baseline_lexical['n_queries'] and frozen_test['mrr'] > floor['mrr']

## 7. Bounded listwise fine-tuning

`pipe.adapt` trains only the last `TRAINABLE_LAYERS` decoder layers — two by default, 31,461,888 of 595,776,512 parameters; the token embeddings (which the output projection shares), the earlier layers and the final norm stay frozen — with a **listwise** loss: each training query contributes its positive and its first `TRAIN_CANDIDATES - 1` negatives, formatted exactly as `rerank` formats them, the relevance logit of every candidate is the `yes`-minus-`no` logit at the last position, and the loss is the cross-entropy of the positive over its list. `BATCH_SIZE` counts queries per step; AdamW at a fixed learning rate, gradient clipping at 1.0, seeded shuffling and no scheduler; prompts are truncated to `MAX_TRAIN_TOKENS` (192) **during training only**. Epoch 0 records the frozen model's validation ranking metrics; every epoch is scored the same way, and the epoch with the highest validation MRR is kept.

Watch validation recall@1 climb over two epochs (about 170 s of training plus a validation pass per epoch on CPU). The build record's sweep on this sample: two layers at 2e-5 reached recall@1 77.9 % (validation MRR still rising at epoch 2); two layers at 5e-5 peaked at epoch 1 and ended at 76.0 % — the default keeps 2e-5.

In [ ]:
EPOCHS = 2  # @param {type:"integer"}
LEARNING_RATE = 2e-5  # @param {type:"number"}
BATCH_SIZE = 4  # @param {type:"integer"}
TRAINABLE_LAYERS = 2  # @param {type:"integer"}
TRAIN_CANDIDATES = 4  # @param {type:"integer"}

def report(entry):
    row = {'epoch': entry['epoch'], 'train_loss': None if entry['train_loss'] is None else round(entry['train_loss'], 4)}
    if entry.get('val'):
        row['val_recall@1'] = round(entry['val']['recall@1'], 4)
        row['val_recall@3'] = round(entry['val']['recall@3'], 4)
        row['val_mrr'] = round(entry['val']['mrr'], 4)
    if 'note' in entry:
        row['note'] = entry['note']
    print(row)

t0 = time.perf_counter()
adapt_result = pipe.adapt(train_records, val_records, instruction=INSTRUCTION, epochs=EPOCHS, lr=LEARNING_RATE, batch_size=BATCH_SIZE, trainable_layers=TRAINABLE_LAYERS, train_candidates=TRAIN_CANDIDATES, progress=report)
adapt_seconds = round(time.perf_counter() - t0, 1)
print({'objective': adapt_result['objective'], 'trainable_parameters': adapt_result['n_trainable'], 'total_parameters': adapt_result['n_total'], 'train_pairs': adapt_result['n_train_pairs'], 'best_epoch': adapt_result['best_epoch'], 'selection': adapt_result['selection'], 'seconds': adapt_seconds})

## 8. Held-out evaluation

The test split was never used for training or epoch selection, and no message in it appears in the training or validation splits. The adapted reranker is scored exactly as the frozen one was in Section 6 — same shortlists, same instruction — and the four rows are put side by side. Look for recall@1 up by several points and MRR up accordingly; the cell asserts the adapted MRR is above the frozen MRR. 154 shortlists from one seeded split of one corpus give no dispersion estimate; the deltas are sample-sanity evidence that the adaptation contract works, not a benchmark, and a gain on Banking77 intent shortlists says nothing about your reranking task until you measure it there. Another 132 s on CPU.

In [ ]:
adapted_test = pipe.evaluate(test_records, instruction=INSTRUCTION)
adapted_val = pipe.evaluate(val_records, instruction=INSTRUCTION)
comparison = {
    metric: {'random_floor': round(floor[metric], 4), 'lexical': round(baseline_lexical[metric], 4), 'frozen': round(frozen_test[metric], 4), 'adapted': round(adapted_test[metric], 4)}
    for metric in ('recall@1', 'recall@3', 'recall@5', 'mrr')
}
comparison['median_rank'] = {'lexical': baseline_lexical['median_rank'], 'frozen': frozen_test['median_rank'], 'adapted': adapted_test['median_rank']}
comparison['delta_vs_frozen'] = {metric: round(adapted_test[metric] - frozen_test[metric], 4) for metric in ('recall@1', 'recall@3', 'recall@5', 'mrr')}
for metric, row in comparison.items():
    print({metric: row})
evaluation_report_payload = {
    'model': {'id': MODEL_ID, 'revision': MODEL_REVISION, 'key': MODEL_KEY},
    'data_source': data_source,
    'instruction': INSTRUCTION,
    'dataset_digests': {name: manifest['digest'] for name, manifest in dataset_manifests.items()},
    'splits': disjoint,
    'candidates_per_query': dataset_manifests['test']['candidates'],
    'baselines': {'random_floor': floor, 'lexical': baseline_lexical},
    'frozen_test': frozen_test,
    'validation_metrics': adapted_val,
    'test_metrics': adapted_test,
    'comparison': comparison,
    'adaptation': {k: v for k, v in adapt_result.items() if k not in ('history', 'trainable_names')},
    'history': adapt_result['history'],
    'adaptation_seconds': adapt_seconds,
}
with open('outputs/qwen3_reranker_evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(evaluation_report_payload, f, indent=2, ensure_ascii=False)
assert adapted_test['mrr'] > frozen_test['mrr']
print({'report': 'outputs/qwen3_reranker_evaluation_report.json'})

## 9. Rerank before and after, export the adapter and reload it

The three test shortlists ranked by the frozen model in Section 5 are ranked again by the adapted model through the same `rerank` contract and printed side by side with the gold intent's rank in each. Read them as observations: the metric is Section 8, and the scores of the adapted model live on a different scale from the frozen ones — the `yes` share moves for every pair, so a score is not comparable across the two models. The per-batch `evaluation_report` helper — the inference-stage helper — is written for the first shortlist and stays `not-measurable`, because a batch of scores has no metric without relevance labels; `pipe.evaluate` is that labelled evaluation.

`pipe.save_artifact` writes the trained tensors — the last two decoder layers, about 126 MB — as `adapter.safetensors`, with a `manifest.json` recording the artifact format, the base model id and revision, the digest of the base `model.safetensors`, the instruction it was trained with, the tensor names, the file size and SHA-256, the training configuration and the epoch history (OUT8). `Qwen3RerankerPipeline.from_artifact` re-verifies the base snapshot, checks the artifact manifest and digest **before** deserialising, refuses any tensor that is not a decoder-layer tensor of the base, and overlays the tensors onto a freshly loaded base — a new object from files, not the in-memory model (VER2). The cell asserts identical scores on the probe shortlists and an identical validation MRR (VER4).

In [ ]:
import csv
import shutil

after = {r['id']: ranked_shortlist(r) for r in probe_records}
rows = []
for r in probe_records:
    rows.append({'id': r['id'], 'query': r['query'], 'gold': r['positive'], 'frozen_order': ' | '.join(d for d, _s in before[r['id']]), 'adapted_order': ' | '.join(d for d, _s in after[r['id']]), 'gold_rank_frozen': next(i + 1 for i, (d, _s) in enumerate(before[r['id']]) if d == r['positive']), 'gold_rank_adapted': next(i + 1 for i, (d, _s) in enumerate(after[r['id']]) if d == r['positive']), 'gold_score_frozen': next(s for d, s in before[r['id']] if d == r['positive']), 'gold_score_adapted': next(s for d, s in after[r['id']] if d == r['positive'])})
    print({k: rows[-1][k] for k in ('id', 'gold', 'frozen_order', 'adapted_order', 'gold_rank_frozen', 'gold_rank_adapted', 'gold_score_frozen', 'gold_score_adapted')})
single_report = evaluation_report(pipe.rerank(pairs, instruction=INSTRUCTION), sample_kind='one Banking77 test shortlist' if not USE_BYOD else 'one BYOD test shortlist')
print({'batch_report_verdict': single_report['verdict'], 'shortlists_reordered': sum(r['frozen_order'] != r['adapted_order'] for r in rows), 'of': len(rows)})
with open('outputs/qwen3_reranker_reranking.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)

artifact_dir = Path('outputs/qwen3_reranker_adapter')
shutil.rmtree(artifact_dir, ignore_errors=True)
pipe.save_artifact(artifact_dir, metadata={'tutorial': 'qwen3_reranker', 'data_source': data_source})
artifact_manifest = json.loads((artifact_dir / 'manifest.json').read_text(encoding='utf-8'))
print({'artifact': str(artifact_dir), 'format': artifact_manifest['format'], 'instruction': artifact_manifest['adapter']['instruction'], 'tensors': len(artifact_manifest['tensors']), 'bytes': artifact_manifest['files'][0]['bytes'], 'sha256': artifact_manifest['files'][0]['sha256'][:16] + '...'})

reloaded = Qwen3RerankerPipeline.from_artifact(artifact_dir, weights_dir=WEIGHTS_DIR, device=pipe.device)
reloaded_scores = reloaded.rerank(pairs, instruction=INSTRUCTION)['scores']
in_memory_scores = pipe.rerank(pairs, instruction=INSTRUCTION)['scores']
reloaded_val = reloaded.evaluate(val_records[:20], instruction=INSTRUCTION)
in_memory_val = pipe.evaluate(val_records[:20], instruction=INSTRUCTION)
parity = {'scores_identical': reloaded_scores == in_memory_scores, 'mrr_in_memory': round(in_memory_val['mrr'], 6), 'mrr_reloaded': round(reloaded_val['mrr'], 6)}
print({'reload_parity': parity, 'reloaded_best_epoch': reloaded.adapter['best_epoch']})
assert parity['scores_identical'] and abs(in_memory_val['mrr'] - reloaded_val['mrr']) < 1e-9

weight_entry = next(entry for entry in snapshot['files'] if entry['path'] == WEIGHT_FILE)
result_payload = {
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'snapshot': {'path': str(WEIGHTS_DIR), 'files': len(snapshot['files']), 'total_bytes': snapshot.get('totalBytes'), 'fetched_this_run': fetched, 'weight_file': WEIGHT_FILE, 'weight_format': 'safetensors, digest-verified', 'weight_sha256': weight_entry['sha256']},
    'data_source': data_source,
    'instruction': INSTRUCTION,
    'corpus': {'name': CORPUS_NAME, 'release': CORPUS_RELEASE, 'base_url': CORPUS_BASE_URL, 'files': {k: {'name': v[0], 'bytes': v[1], 'sha256': v[2]} for k, v in CORPUS_FILES.items()}, 'license': CORPUS_LICENSE},
    'inference_contract': {'input_manifest': input_manifest, 'sanity_checks': checks, 'query': record['query'], 'candidates': candidates, 'scores': scores, 'ranking': result['ranking'], 'n_tokens': result['n_tokens'], 'seconds': rerank_seconds},
    'comparison': comparison,
    'reranking_before_after': rows,
    'batch_report': single_report,
    'artifact': {'dir': str(artifact_dir), 'sha256': artifact_manifest['files'][0]['sha256'], 'bytes': artifact_manifest['files'][0]['bytes'], 'tensors': len(artifact_manifest['tensors'])},
    'reload_parity': parity,
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'device': pipe.device, 'dtype': 'bfloat16' if pipe.device.startswith('cuda') else 'float32'},
}
with open('outputs/qwen3_reranker_result.json', 'w', encoding='utf-8') as handle:
    json.dump(result_payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The frozen reranker already puts the right intent phrase first for most unseen messages (recall@1 in the seventies against a lexical baseline that the hard negatives were chosen to defeat and a random floor of 16.7 %), and a bounded listwise fine-tuning of the last two decoder layers on 231 shortlists lifts held-out recall@1 by +4.5 points in a few minutes on CPU, with a 126 MB adapter that reloads to identical scores. That is the claim: the adaptation contract can adapt the reranker to a relevance judgement end to end on a real labelled corpus, and the numbers it produces are read against a random floor, a lexical baseline and the frozen model rather than in isolation.

The test split is 154 six-entry shortlists from one seeded split of one corpus with no dispersion estimate; recall@k and MRR say whether the gold document is ordered first within a shortlist the dataset supplied, not how the model would rank a real candidate pool. The adapter changes the last layers, which every pair shares, so every score shifts and scores are not comparable across the frozen and adapted models; nothing here measures the effect on other judgements. On CUDA the model runs in bfloat16 and the recorded float32 CPU numbers will not reproduce to the last digit. A cross-encoder scores one pair per forward pass, so evaluation cost grows with the shortlist length — a first-stage retriever (the sibling embedding pipeline) is what makes the shortlist short.

Three things to carry to real data. **Floors first:** the random floor and the lexical baseline on *your* shortlists are the numbers to read before any reranker's; a lexical baseline near the frozen model means the task is mostly keyword matching. **Leakage:** de-duplicate queries across splits (the contract does this case-insensitively) and split by user or session when your queries come from one. **Negatives:** the quality of the shortlist decides what the reranker learns — draw negatives from the retriever you will deploy behind, not at random.

Successful execution proves that the recorded repository revision's pipeline modules, carried in this standalone notebook, can acquire and digest-verify the pinned model snapshot, fetch and digest-verify a real labelled corpus, validate the demonstrated dataset contract without leakage, execute the reranking contract and a bounded listwise fine-tuning, evaluate by ranking metrics against a random floor, a lexical baseline and the frozen model on an independent split, and emit the shown machine-readable artifacts — without the repository being reachable. It does **not** establish benchmark superiority, ranking quality on any other task, a usable acceptance threshold, or production fitness.

**Optional experiments (they do not affect the default path):** set `TRAINABLE_LAYERS = 1` and watch the gain shrink; set `TRAIN_CANDIDATES = 6` to train on the whole shortlist and compare; set `EPOCHS = 3` and watch whether validation MRR keeps rising or turns (the best epoch is kept either way); change `INSTRUCTION` and re-read the frozen numbers — the judgement is instruction-conditioned; or bring your own shortlists through BYOD and read the lexical baseline before the adapted number.

## References

- Repository README: https://github.com/kurtvalcorza/qwen3-reranker-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/qwen3-reranker-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/qwen3-reranker-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/Qwen/Qwen3-Reranker-0.6B
- Upstream code: https://github.com/QwenLM/Qwen3-Embedding
- Qwen3 Embedding: Advancing Text Embedding and Reranking Through Foundation Models (2025): https://arxiv.org/abs/2506.05176
- Efficient Intent Detection with Dual Sentence Encoders (Casanueva et al., 2020; Banking77, CC BY 4.0): https://arxiv.org/abs/2003.04807
- DIMER Notebook Specification 2.0 and Model Card Specification 1.1 (fleet specs in the ml-worker repository)